# ❤️ Heart Disease Prediction & Analysis
### A Step-by-Step Machine Learning Project for Beginners

---

**Dataset:** Heart Disease UCI (from Kaggle)  
**Goal:** Predict whether a patient has heart disease based on medical data  

### 📋 What We'll Do in This Notebook:
1. Load and understand the dataset
2. Explore and visualize the data
3. Clean and prepare the data
4. Build Machine Learning models to predict heart disease
5. Evaluate and compare our models

---
## 📦 Step 1: Install & Import Libraries

Think of libraries as **toolboxes**. Each one gives us special tools:
- `pandas` → work with tables (like Excel)
- `numpy` → math operations
- `matplotlib` / `seaborn` → make charts
- `sklearn` → machine learning tools

In [1]:
# 📦 Import all the libraries we need
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

#Machine Learning tools
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)


#Make Charts Look Nice
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print("All libraries imported successfully!")


All libraries imported successfully!


---
## 📂 Step 2: Load the Dataset

We're using the **Heart Disease UCI dataset** — a classic medical dataset.

### About the Dataset:
- **303 patients** with 14 columns of medical information
- Each row = one patient
- **Target column:** `target` — 1 means has heart disease, 0 means no heart disease

### Column Meanings (Plain English):
| Column | Meaning |
|--------|----------|
| age | Patient's age |
| sex | 1 = Male, 0 = Female |
| cp | Chest pain type (0–3) |
| trestbps | Resting blood pressure |
| chol | Cholesterol level |
| fbs | Fasting blood sugar > 120 mg/dl (1=Yes) |
| restecg | Resting ECG results (0–2) |
| thalach | Maximum heart rate achieved |
| exang | Exercise induced angina (1=Yes) |
| oldpeak | ST depression value |
| slope | Slope of peak exercise ST segment |
| ca | Number of major vessels colored by fluoroscopy |
| thal | Thalassemia blood disorder type |
| target | 1 = Heart disease, 0 = No heart disease |

In [7]:
# Load the dataset
import pandas as pd

try:
    df = pd.read_csv("heart_disease.csv")
    print("✅ Dataset Loaded Successfully!")

except Exception as e:
    # Fallback: create sample data if file fails to load
    print(f"⚠️ Loading failed: {e}")
    print("Using built-in sample data...")

    from sklearn.datasets import make_classification

    X, y = make_classification(
        n_samples=303,
        n_features=14,
        random_state=42
    )

    cols = [
        'age', 'sex', 'cp', 'trestbps', 'chol',
        'fbs', 'restecg', 'thalach', 'exang',
        'oldpeak', 'slope', 'ca', 'thal',
        'extra_feature'
    ]

    df = pd.DataFrame(X, columns=cols)
    df['target'] = y
    df['age'] = (df['age'] * 15 + 55).astype(int).clip(25, 80)

# Showing the first five rows of the dataset
print(f"\nDataset Shape: {df.shape[0]} rows x {df.shape[1]} columns")
print("\nFirst 5 rows of the dataset:")

print(df.head())  

✅ Dataset Loaded Successfully!

Dataset Shape: 303 rows x 14 columns

First 5 rows of the dataset:
   age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  slope  \
0   63    1   3       145   233    1        0      150      0      2.3      0   
1   37    1   2       130   250    0        1      187      0      3.5      0   
2   41    0   1       130   204    0        0      172      0      1.4      2   
3   56    1   1       120   236    0        1      178      0      0.8      2   
4   57    0   0       120   354    0        1      163      1      0.6      2   

   ca  thal  target  
0   0     1       1  
1   0     2       1  
2   0     2       1  
3   0     2       1  
4   0     2       1  


---
## 🔍 Step 3: Understand the Data

Before doing anything, we need to **understand** what we're working with.
Think of this like reading a document before writing about it.

In [8]:
#BAsic info from the dataset
print('==== Dataset Overview ====')
print(f'Total patients: {len(df)}')
print(f'Total Columns: {len(df.columns)}')
print(f'\n Column Names: {list(df.columns)}')

print('\n=== Data Types ===')
print(df.dtypes)

print('\n=== ANY MISSING VALUES? ===')
missing = df.isnull().sum()
if missing.sum() == 0:
    print("Great News! No missing values found.")
else:
    print(missing[missing > 0])
    

==== Dataset Overview ====
Total patients: 303
Total Columns: 14

 Column Names: ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']

=== Data Types ===
age           int64
sex           int64
cp            int64
trestbps      int64
chol          int64
fbs           int64
restecg       int64
thalach       int64
exang         int64
oldpeak     float64
slope         int64
ca            int64
thal          int64
target        int64
dtype: object

=== ANY MISSING VALUES? ===
Great News! No missing values found.


In [ ]:
#Stastical Summary of teh dataset
print('==== SUMMARY STATISTICS ====')
print('(This shows average, min, max, and quartiles for each numeric column)')
df.describe().round(2)

==== SUMMARY STATISTICS ====
(This shows average, min, max values for each column)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
count,303.00,303.00,303.00,303.00,303.00,303.00,303.00,303.00,303.00,303.00,303.00,303.00,303.00,303.00
mean,54.37,0.68,0.97,131.62,246.26,0.15,0.53,149.65,0.33,1.04,1.40,0.73,2.31,0.54
std,9.08,0.47,1.03,17.54,51.83,0.36,0.53,22.91,0.47,1.16,0.62,1.02,0.61,0.50
min,29.00,0.00,0.00,94.00,126.00,0.00,0.00,71.00,0.00,0.00,0.00,0.00,0.00,0.00
25%,47.50,0.00,0.00,120.00,211.00,0.00,0.00,133.50,0.00,0.00,1.00,0.00,2.00,0.00
50%,55.00,1.00,1.00,130.00,240.00,0.00,1.00,153.00,0.00,0.80,1.00,0.00,2.00,1.00
75%,61.00,1.00,2.00,140.00,274.50,0.00,1.00,166.00,1.00,1.60,2.00,1.00,3.00,1.00
max,77.00,1.00,3.00,200.00,564.00,1.00,2.00,202.00,1.00,6.20,2.00,4.00,3.00,1.00
